<a href="https://colab.research.google.com/github/Sebi2005/Metaheuristics/blob/main/notebooks/ParticleSwarmOptimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1. Problem Definition: Sphere Function**

We continue using the Sphere Function as the baseline for implementation:

Formula: $f(x) = \sum_{i=1}^{n} x_i^2$

Search Space: $-5.12 \le x_i \le 5.12$

Global Optimum: $f(0,0,...,0) = 0$

In [ ]:
import random
import math

def fitness_sphere(sol):
    return sum(x**2 for x in sol)

**2. PSO Implementation Components**

a. Particle Initialization

Each particle needs a position (current solution), a velocity (direction and speed of movement), and a memory of its personal best ($pBest$).


In [ ]:
def initialize_particles(swarm_size, dimensions, low = -5.12, high = 5.12):
  swarm = []
  for _ in range(swarm_size):
    position = [random.uniform(low, high) for _ in range(dimensions)]
    velocity = [random.uniform(0, 1) for _ in range(dimensions)]
    swarm.append({
        'pos':position,
        'vel':velocity,
        'pbest_pos':list(position),
        'pbest_fit':fitness_sphere(position)
    })
  return swarm

b. Velocity and Position Update

 The velocity is updated based on three factors:

 Inertia: Maintaining the current direction.

 Cognitive: Moving toward its own personal best ($pBest$).

 Social: Moving toward the swarm's global best ($gBest$).

In [ ]:
def update_particle(particle, gbest_pos, w = 1, c1 = 2, c2 = 2):
  for i in range(len(particle['pos'])):
    r1, r2 = random.random(), random.random()
    particle['vel'][i] = (w * particle['vel'][i] +
                          c1 * r1 * (particle['pbest_pos'][i] - particle['pos'][i]) +
                          c2 * r2 * (gbest_pos[i] - particle['pos'][i]))
    particle['pos'][i] += particle['vel'][i]
    particle['pos'][i] = max(min(particle['pos'][i], 5.12), -5.12)



In [ ]:
def pso_sphere(swarm_size, dimensions, iterations, w = 1, c1 = 2, c2 = 2):
  swarm = initialize_particles(swarm_size, dimensions)

  gbest_particle = min(swarm, key = lambda p : p['pbest_fit'])
  gbest_pos = list(gbest_particle['pbest_pos'])
  gbest_fit = gbest_particle['pbest_fit']

  history = []

  for _ in range(iterations):
    for particle in swarm:
      update_particle(particle, gbest_pos, w, c1, c2)
      current_fit = fitness_sphere(particle['pos'])
      if current_fit < particle['pbest_fit']:
        particle['pbest_fit'] = current_fit
        particle['pbest_pos'] = list(particle['pos'])

        if current_fit < gbest_fit:
              gbest_fit = current_fit
              gbest_pos = list(particle['pos'])
    history.append(gbest_fit)
  return gbest_pos, gbest_fit, history


In [ ]:
def test_pso_parameters(dimensions=5, iterations=100):
    test_configs = [
        {"name": "Small Swarm, High Inertia", "size": 10, "w": 0.9, "c1": 1.5, "c2": 1.5},
        {"name": "Large Swarm, Low Inertia", "size": 50, "w": 0.4, "c1": 1.5, "c2": 1.5},
        {"name": "Balanced Swarm", "size": 30, "w": 0.7, "c1": 2.0, "c2": 2.0}
    ]

    print(f"{'Configuration':<30} | {'Best Fitness':<20}")
    print("-" * 55)

    for cfg in test_configs:
        best_pos, best_fit, history = pso_sphere(
            swarm_size=cfg['size'],
            dimensions=dimensions,
            iterations=iterations,
            w=cfg['w'],
            c1=cfg['c1'],
            c2=cfg['c2']
        )

        print(f"{cfg['name']:<30} | {best_fit:<20.10f}")

test_pso_parameters()

Configuration                  | Best Fitness        
-------------------------------------------------------
Small Swarm, High Inertia      | 0.2111902361        
Large Swarm, Low Inertia       | 0.0000000000        
Balanced Swarm                 | 0.0000422253        


## Comparing two variants of PSO on the Sphere Function and the Schwefel Functions

**1.1 Problem Definition**

Schwefel Function ($f_7$): A deceptive, multimodal function where $f(x) = \sum -x_i \cdot \sin(\sqrt{|x_i|})$.

(Search space: $[-500, 500]$)

In [ ]:
def fitness_schwefel(sol):
  return sum(-x * math.sin(math.sqrt(abs(x))) for x in sol)

**2. Implementation of PSO Variants**

**Variant 1: Inertia Weight Decay PSO**

This variant uses a linearly decreasing weight $w$ to transition from exploration (searching wide) to exploitation (fine-tuning near the best found point).

$$w = w_{max} - \frac{iteration}{max\_iterations} \cdot (w_{max} - w_{min})$$



In [ ]:
def update_velocity_inertia(particle, gbest_pos, w, c1=2.05, c2=2.05):
    for i in range(len(particle['pos'])):
        r1, r2 = random.random(), random.random()
        particle['vel'][i] = (w * particle['vel'][i] +
                              c1 * r1 * (particle['pbest_pos'][i] - particle['pos'][i]) +
                              c2 * r2 * (gbest_pos[i] - particle['pos'][i]))
    return particle['vel']

**Variant 2: Constriction Factor PSO**

This variant uses a mathematical multiplier, $\chi$ (chi), applied to the entire velocity calculation. It is designed to prevent the "explosion" of velocity and mathematically guarantees that the swarm will eventually converge to a stable point.

$$V_{i}(t+1) = \chi \cdot [V_{i}(t) + c_1 \cdot r_1 \cdot (pBest_{i} - X_{i}(t)) + c_2 \cdot r_2 \cdot (gBest - X_{i}(t))]$$


$\chi$ (Constriction Factor): Calculated using the formula $\chi = \frac{2}{|2 - \phi - \sqrt{\phi^2 - 4\phi}|}$, where $\phi = c_1 + c_2$ and $\phi > 4$

In [ ]:
def update_velocity_constriction(particle, gbest_pos, chi, c1=2.05, c2=2.05):
    for i in range(len(particle['pos'])):
        r1, r2 = random.random(), random.random()
        particle['vel'][i] = chi * (particle['vel'][i] +
                                   c1 * r1 * (particle['pbest_pos'][i] - particle['pos'][i]) +
                                   c2 * r2 * (gbest_pos[i] - particle['pos'][i]))
    return particle['vel']

**3. The Assignment A7 Orchestrator**

This function is designed to handle multiple functions, parameters, and variants of the experiments.

In [ ]:
def evolutionary_algorithm_pso_A7(variant_name, fitness_func, bounds,
                                  swarm_size=30, dimensions=5, iterations=200):

    low, high = bounds
    swarm = initialize_particles(swarm_size, dimensions, low, high)

    gbest_particle = min(swarm, key=lambda p: p['pbest_fit'])
    gbest_pos = list(gbest_particle['pbest_pos'])
    gbest_fit = gbest_particle['pbest_fit']

    phi = 4.1
    chi = 2.0 / abs(2.0 - phi - math.sqrt(phi**2 - 4.0 * phi))

    for t in range(iterations):
        w = 0.9 - (t / iterations) * (0.9 - 0.4) # for intertia weight decay pso
        for particle in swarm:
            if variant_name == "Inertia_Decay":
                particle['vel'] = update_velocity_inertia(particle, gbest_pos, w)
            else:
                particle['vel'] = update_velocity_constriction(particle, gbest_pos, chi)

            for i in range(dimensions):
                particle['pos'][i] += particle['vel'][i]
                particle['pos'][i] = max(min(particle['pos'][i], high), low)

            current_fit = fitness_func(particle['pos'])

            if current_fit < particle['pbest_fit']:
                particle['pbest_fit'] = current_fit
                particle['pbest_pos'] = list(particle['pos'])

                if current_fit < gbest_fit:
                    gbest_fit = current_fit
                    gbest_pos = list(particle['pos'])

    return gbest_fit

**4. Comparative Experiments**

In [ ]:
import csv
def perform_A7_experiments():
    problems = [
        {"name": "Sphere", "func": fitness_sphere, "bounds": (-5.12, 5.12)},
        {"name": "Schwefel", "func": fitness_schwefel, "bounds": (-500, 500)}
    ]
    variants = ["Inertia_Decay", "Constriction"]
    results = []

    for prob in problems:
        for var in variants:
            print(f"Testing {prob['name']} with {var}...")
            runs = [evolutionary_algorithm_pso_A7(var, prob['func'], prob['bounds']) for _ in range(10)]

            results.append({
                "Problem": prob['name'],
                "Variant": var,
                "Best_Fitness": min(runs),
                "Avg_Fitness": sum(runs) / len(runs),
                "Runs": 10
            })


    with open('A7_PSO_Comparison.csv', 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)
perform_A7_experiments()

Testing Sphere with Inertia_Decay...
Testing Sphere with Constriction...
Testing Schwefel with Inertia_Decay...
Testing Schwefel with Constriction...


|Problem|Variant|Best\_Fitness|Avg\_Fitness|Runs|
|---|---|---|---|---|
|Sphere|Inertia\_Decay|1\.0665458811411905e-10|6\.45157240189701e-10|10|
|Sphere|Constriction|7\.537812277126178e-16|9\.610961334419085e-15|10|
|Schwefel|Inertia\_Decay|-1856\.5207076114486|-1607\.0416751711555|10|
|Schwefel|Constriction|-1976\.476101747678|-1691\.0104510633107|10|



**Sphere Function**: High Precision

The target for the Sphere function is 0.

Inertia Decay: Achieved a fitness of $10^{-10}$, which is extremely close to zero.

Constriction Factor: Achieved $10^{-16}$. This is essentially "machine zero," indicating that the Constriction variant is mathematically more stable for convex, unimodal functions.

Verdict: These results are really good. They show that the swarm is effectively collapsing toward the global optimum without overshooting it.


**Schwefel Function**: Strong Convergence

For 5 dimensions ($d=5$), the theoretical global optimum is roughly $-2094.91$.

Best Fitness: The "Constriction" variant hit $-1976.47$.

Interpretation: Reaching $-1976$ means the particles found the global valley. Because Schwefel is "multimodal" and "deceptive," many algorithms get trapped in local minima around $-1200$ or $-1600$.

Verdict: These are very good results for a standard PSO. It proves swarm has enough "velocity" to escape local traps but enough "control" to settle near the true bottom.

**Inertia Decay vs. Constriction**

Sphere: Constriction was significantly more precise ($10^{-16}$ vs $10^{-10}$). This is because the $\chi$ (chi) factor mathematically guarantees convergence by dampening the velocity more effectively than simple linear decay.

Schwefel: Constriction also performed better here ($-1976$ vs $-1856$). This suggests that the Constriction variant maintains a better balance between exploring the wide $[-500, 500]$ space and exploiting the final valley.